In [ ]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive')

# Add src to path
project_path = '/content/drive/MyDrive/Digital-Warranty-Organizer'
sys.path.append(os.path.join(project_path, 'src'))

# Professional Imports
import fraud_utils
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib # For saving the model

print("✅ Fraud detection environment ready.")

Mounted at /content/drive
✅ Fraud detection environment ready.


Load and Prepare Data

In [ ]:
# Load the raw data
raw_data_path = os.path.join(project_path, 'data', 'raw', 'train.csv')
df_raw = pd.read_csv(raw_data_path)

# Process using your modular function
df_clean = fraud_utils.preprocess_fraud_data(df_raw)

# Split features (X) and target (y)
X = df_clean.drop('Fraud', axis=1)
y = df_clean['Fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Data processed. Features: {X.shape[1]}")
print(f"Training on {len(X_train)} samples.")

✅ Data processed. Features: 19
Training on 6672 samples.


In [ ]:
# =========================
# STEP 4: FIX src/fraud_utils.py
# =========================

import os

fraud_utils_path = os.path.join(project_path, "src", "fraud_utils.py")

new_code = r'''
import os
import joblib
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import IsolationForest


def calculate_fraud_weight(y):
    """
    Calculates class imbalance weight for XGBoost.
    Fraud class = 1, Normal class = 0.
    """
    fraud_count = (y == 1).sum()
    normal_count = (y == 0).sum()

    if fraud_count == 0:
        return 1.0

    return normal_count / fraud_count


def preprocess_fraud_data(df, encoder_path=None, is_training=True):
    """
    Cleans data and encodes categorical columns.

    Training:
        - Fits LabelEncoders
        - Saves encoders to encoder_path

    Inference:
        - Loads saved encoders
        - Unknown categories become -1
    """
    df = df.copy()

    if encoder_path:
        os.makedirs(encoder_path, exist_ok=True)

    if "Unnamed: 0" in df.columns:
        df = df.drop("Unnamed: 0", axis=1)

    df = df.fillna(0)

    categorical_cols = df.select_dtypes(include=["object"]).columns

    for col in categorical_cols:
        le_file = os.path.join(encoder_path, f"le_{col}.pkl") if encoder_path else None

        if is_training:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

            if le_file:
                joblib.dump(le, le_file)

        else:
            if not le_file or not os.path.exists(le_file):
                raise FileNotFoundError(f"Missing encoder for column '{col}': {le_file}")

            le = joblib.load(le_file)

            def safe_transform(value):
                value = str(value)
                if value in le.classes_:
                    return int(le.transform([value])[0])
                return -1

            df[col] = df[col].apply(safe_transform)

    return df


def get_hybrid_features(df, model_path=None, is_training=True):
    """
    Adds IsolationForest anomaly score as an extra feature.
    anomaly_score:
        0 = normal
        1 = anomaly/suspicious
    """
    df = df.copy()

    if model_path:
        os.makedirs(model_path, exist_ok=True)

    iso_file = os.path.join(model_path, "iso_forest.pkl") if model_path else None

    features = df.drop("Fraud", axis=1) if "Fraud" in df.columns else df

    if is_training:
        iso = IsolationForest(
            n_estimators=100,
            contamination=0.1,
            random_state=42
        )

        raw_scores = iso.fit_predict(features)

        if iso_file:
            joblib.dump(iso, iso_file)

    else:
        if not iso_file or not os.path.exists(iso_file):
            raise FileNotFoundError(f"Missing IsolationForest model: {iso_file}")

        iso = joblib.load(iso_file)
        raw_scores = iso.predict(features)

    df["anomaly_score"] = pd.Series(raw_scores).map({1: 0, -1: 1}).values

    return df
'''

# Backup old file
backup_path = fraud_utils_path + ".backup"
if os.path.exists(fraud_utils_path):
    with open(fraud_utils_path, "r", encoding="utf-8") as f:
        old_code = f.read()
    with open(backup_path, "w", encoding="utf-8") as f:
        f.write(old_code)

# Write fixed file
with open(fraud_utils_path, "w", encoding="utf-8") as f:
    f.write(new_code)

print("✅ fraud_utils.py fixed")
print("Backup saved to:", backup_path)
print("Updated file:", fraud_utils_path)

✅ fraud_utils.py fixed
Backup saved to: /content/drive/MyDrive/Digital-Warranty-Organizer/src/fraud_utils.py.backup
Updated file: /content/drive/MyDrive/Digital-Warranty-Organizer/src/fraud_utils.py


In [ ]:
# =========================
# STEP 4 CHECK
# =========================

import importlib
import fraud_utils

importlib.reload(fraud_utils)

print("calculate_fraud_weight exists:", hasattr(fraud_utils, "calculate_fraud_weight"))
print("preprocess_fraud_data exists:", hasattr(fraud_utils, "preprocess_fraud_data"))
print("get_hybrid_features exists:", hasattr(fraud_utils, "get_hybrid_features"))

print("✅ Step 4 completed")

calculate_fraud_weight exists: True
preprocess_fraud_data exists: True
get_hybrid_features exists: True
✅ Step 4 completed


In [ ]:
# =========================
# STEP 5: TRAIN + SAVE HYBRID FRAUD MODEL ASSETS
# =========================

import os
import importlib
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

import fraud_utils
importlib.reload(fraud_utils)

# 1. Paths
model_dir = os.path.join(project_path, "models", "fraud_assets")
os.makedirs(model_dir, exist_ok=True)

raw_data_path = os.path.join(project_path, "data", "raw", "train.csv")

# 2. Load data
df_raw = pd.read_csv(raw_data_path)
print("Raw data shape:", df_raw.shape)

# 3. Preprocess and save label encoders
df_clean = fraud_utils.preprocess_fraud_data(
    df_raw,
    encoder_path=model_dir,
    is_training=True
)

print("Clean data shape:", df_clean.shape)

# 4. Add Isolation Forest anomaly score and save iso_forest.pkl
df_hybrid = fraud_utils.get_hybrid_features(
    df_clean,
    model_path=model_dir,
    is_training=True
)

print("Hybrid data shape:", df_hybrid.shape)
print("Columns:", list(df_hybrid.columns))

# 5. Split X and y
X = df_hybrid.drop("Fraud", axis=1)
y = df_hybrid["Fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 6. Calculate fraud class weight
scale_w = fraud_utils.calculate_fraud_weight(y_train)
print("scale_pos_weight:", scale_w)

# 7. Train XGBoost
fraud_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=scale_w,
    eval_metric="logloss",
    random_state=42
)

print("🚀 Training hybrid fraud model...")
fraud_model.fit(X_train, y_train)

# 8. Evaluate
y_pred = fraud_model.predict(X_test)

print("\n--- CONFUSION MATRIX ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test, y_pred))

# 9. Save model and feature columns
model_path = os.path.join(model_dir, "xgb_fraud_model.pkl")
features_path = os.path.join(model_dir, "feature_columns.pkl")

joblib.dump(fraud_model, model_path)
joblib.dump(list(X.columns), features_path)

print("\n✅ Fraud model saved to:", model_path)
print("✅ Feature columns saved to:", features_path)
print("✅ Fraud assets folder:", model_dir)

Raw data shape: (8341, 21)
Clean data shape: (8341, 20)
Hybrid data shape: (8341, 21)
Columns: ['Region', 'State', 'Area', 'City', 'Consumer_profile', 'Product_category', 'Product_type', 'AC_1001_Issue', 'AC_1002_Issue', 'AC_1003_Issue', 'TV_2001_Issue', 'TV_2002_Issue', 'TV_2003_Issue', 'Claim_Value', 'Service_Centre', 'Product_Age', 'Purchased_from', 'Call_details', 'Purpose', 'Fraud', 'anomaly_score']
scale_pos_weight: 11.517823639774859
🚀 Training hybrid fraud model...

--- CONFUSION MATRIX ---
[[1453   83]
 [   0  133]]

--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

           0       1.00      0.95      0.97      1536
           1       0.62      1.00      0.76       133

    accuracy                           0.95      1669
   macro avg       0.81      0.97      0.87      1669
weighted avg       0.97      0.95      0.96      1669


✅ Fraud model saved to: /content/drive/MyDrive/Digital-Warranty-Organizer/models/fraud_assets/xgb_fraud_model.

In [ ]:
# =========================
# STEP 5 CHECK: VERIFY FRAUD ASSETS
# =========================

import os

model_dir = os.path.join(project_path, "models", "fraud_assets")

print("Folder exists:", os.path.exists(model_dir))
print("Files inside:")
for f in os.listdir(model_dir):
    print("-", f)

Folder exists: True
Files inside:
- le_Region.pkl
- le_State.pkl
- le_Area.pkl
- le_City.pkl
- le_Consumer_profile.pkl
- le_Product_category.pkl
- le_Product_type.pkl
- le_Purchased_from.pkl
- le_Purpose.pkl
- iso_forest.pkl
- xgb_fraud_model.pkl
- feature_columns.pkl


In [ ]:
# =========================
# CHECK inference_engine.py UPDATE
# Run this in the existing fraud notebook
# =========================

import os
import sys
import importlib

project_path = "/content/drive/MyDrive/Digital-Warranty-Organizer"
src_path = os.path.join(project_path, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

import inference_engine
importlib.reload(inference_engine)

print("✅ inference_engine.py reloaded successfully")

✅ inference_engine.py reloaded successfully


In [ ]:
# =========================
# SETUP AFTER RESTART
# =========================

!pip install -q paddleocr paddlepaddle-gpu langchain-text-splitters langchain-core
!pip install -q transformers[torch] datasets evaluate seqeval accelerate xgboost joblib

print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.8/120.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/97

In [ ]:
import os
import sys
from google.colab import drive

drive.mount('/content/drive')

project_path = "/content/drive/MyDrive/Digital-Warranty-Organizer"
src_path = os.path.join(project_path, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from master_inference import WarrantySystem

system = WarrantySystem(project_path)

print("✅ WarrantySystem initialized successfully")
print("Has assess_risk:", hasattr(system, "assess_risk"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

✅ WarrantySystem initialized successfully
Has assess_risk: True


In [ ]:
# =========================
# STEP 7A: TEST FRAUD RISK ONLY
# =========================

dummy_extracted = {
    "COMPANY": "TEST SHOP",
    "DATE": "25/12/2018",
    "ADDRESS": "TEST ADDRESS",
    "TOTAL": "25000",
    "TOTAL_NUMERIC": 25000.0,
    "CONFIDENCE": 0.90
}

risk = system.assess_risk(dummy_extracted, region="North")

print("Fraud risk:", risk)
print("Fraud risk %:", f"{risk * 100:.2f}%")

if risk > 0.5:
    print("STATUS: REJECTED")
else:
    print("STATUS: APPROVED")

Fraud risk: 0.00015189696568995714
Fraud risk %: 0.02%
STATUS: APPROVED


In [ ]:
# =========================
# STEP 7B: TEST NER + FRAUD TOGETHER WITHOUT PADDLEOCR
# Uses existing SROIE OCR box files
# =========================

import os
import shutil
from data_loader import process_receipt

raw_zip = os.path.join(project_path, "data", "raw", "SROIE_Data.zip")
temp_extract_path = "/content/sroie_temp"

if not os.path.exists(temp_extract_path):
    os.makedirs(temp_extract_path, exist_ok=True)
    !unzip -q "{raw_zip}" -d "{temp_extract_path}"
    print("✅ SROIE data unzipped")
else:
    print("ℹ️ SROIE data already exists")

# Use one test receipt
test_img_dir = os.path.join(temp_extract_path, "sroie", "test", "img")
test_box_dir = os.path.join(temp_extract_path, "sroie", "test", "box")
test_ent_dir = os.path.join(temp_extract_path, "sroie", "test", "entities")

image_files = sorted([f for f in os.listdir(test_img_dir) if f.endswith(".jpg")])

sample_file = image_files[0]
sample_name = sample_file.replace(".jpg", "")

image_path = os.path.join(test_img_dir, sample_file)
box_path = os.path.join(test_box_dir, sample_name + ".txt")
entity_path = os.path.join(test_ent_dir, sample_name + ".txt")

sample = process_receipt(
    image_path=image_path,
    ocr_path=box_path,
    label_path=entity_path,
    apply_aug=False
)

print("Sample file:", sample_file)
print("Token count:", len(sample["tokens"]))

# Run NER extraction
extracted = system.extract_info_logic(
    sample["tokens"],
    sample["bboxes"]
)

print("\n--- NER EXTRACTION RESULT ---")
print(extracted)

# Run fraud risk
risk = system.assess_risk(extracted, region="North")

print("\n--- FRAUD RISK RESULT ---")
print("Fraud risk:", risk)
print("Fraud risk %:", f"{risk * 100:.2f}%")

if risk > 0.5:
    print("STATUS: REJECTED")
else:
    print("STATUS: APPROVED")

ℹ️ SROIE data already exists
Sample file: X00016469670.jpg
Token count: 111

--- NER EXTRACTION RESULT ---
{'COMPANY': 'ojc marketing sdn bhd', 'DATE': ': 15 / 01 / 2019 11 : 05 : 16 am', 'ADDRESS': '', 'TOTAL': '193.00', 'TOTAL_NUMERIC': 193.0, 'CONFIDENCE': 0.9983079872633281}

--- FRAUD RISK RESULT ---
Fraud risk: 4.381853068480268e-05
Fraud risk %: 0.00%
STATUS: APPROVED
